Guardrails will:

Fill in your prompt with doctors_notes

Call the LLM using the function you provided (openai.chat.completions.create)

Validate the output against PatientInfo

Fix/reask if needed

Return a result (res) that includes validated output (and metadata depending on version)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # loads .env into environment variables

os.environ["OPENAI_API_KEY"]= os.getenv('OPENAI_API_KEY')
os.environ["GUARDRAILS_API_KEY"]= os.getenv('GUARDRAILS_API_KEY')

In [ ]:
!guardrails hub install hub://guardrails/valid_range

### 1) Imports: what tools in the project

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from guardrails.hub import ValidRange
import guardrails as gd
import openai
import json 
from rich import print

In [ ]:
class Symptom(BaseModel):
    symptom: str = Field(description="Symptom that a patient is experiencing")
    affected_area: str = Field(description="What part of the body the symptom is affecting", validators=[ValidChoices(choices=['head', 'neck', 'chest'], on_fail="reask")])

class Medication(BaseModel):
    medication: str = Field(description="Name of the medication the patient is taking")
    response: str = Field(description="How the patient is responding to the medication")


class PatientInfo(BaseModel):
    gender: str = Field(description="Patient's gender")
    age: int = Field(validators=[ValidRange(min=0, max=100, on_fail="fix")])
    symptoms: List[Symptom] = Field(description="Symptoms that the patient is currently experiencing. Each symptom should be classified into a separate item in the list.")
    current_meds: List[Medication] = Field(description="Medications the patient is currently taking and their response")


In [ ]:
!guardrails configure